# Força bruta e busca exaustiva — Tutorial

**Algoritmos e Estruturas de Dados II (COMP0498) — UFS — 2026.2**

## Objetivos

Ao final deste tutorial você será capaz de:

- Implementar algoritmos de força bruta em C e **medir** o trabalho que eles realizam;
- Construir a entrada de **pior caso** do casamento de padrões ingênuo e confirmar o limite $m(n-m+1)$;
- Percorrer os $2^n$ subconjuntos de $n$ itens com **máscara de bits** e resolver a mochila 0/1 exatamente;
- Percorrer as $(n-1)!$ rotas do caixeiro viajante e **sentir na prática** o salto fatorial;
- Usar a força bruta como **oráculo** para flagrar o erro de um algoritmo guloso.

> As células de código escrevem programas em C com `%%writefile` e depois compilam com `gcc`.
> Rode as células **na ordem**.

In [ ]:
# Verifique se o gcc está disponível no seu ambiente
!gcc --version | head -1


## 1. Casamento de padrões ingênuo

O algoritmo testa **todos** os $n-m+1$ alinhamentos do padrão sobre o texto. Em cada
alinhamento, compara da esquerda para a direita até falhar ou casar por inteiro.

Vamos instrumentar o código com um contador de comparações de caracteres para
comparar dois cenários:

- **texto natural** — as falhas acontecem quase sempre na primeira letra;
- **pior caso** — $T = \texttt{aaa}\ldots\texttt{a}$ e $P = \texttt{aaab}$: cada alinhamento
  casa $m-1$ letras antes de falhar, atingindo exatamente $m(n-m+1)$ comparações.

In [ ]:
%%writefile casamento.c
#include <stdio.h>
#include <string.h>

long comparacoes;                   /* instrumentacao: comparacoes de caractere */

int casamento(const char *T, int n, const char *P, int m) {
    comparacoes = 0;
    for (int i = 0; i <= n - m; i++) {      /* todos os alinhamentos */
        int j = 0;
        while (j < m) {
            comparacoes++;
            if (P[j] != T[i + j]) break;    /* falhou neste alinhamento */
            j++;
        }
        if (j == m) return i;               /* casou por inteiro */
    }
    return -1;
}

int main(void) {
    const char *T = "NOBODY_NOTICED";
    const char *P = "NOT";
    int pos = casamento(T, (int) strlen(T), P, (int) strlen(P));
    printf("texto natural: posicao %d com %ld comparacoes\n", pos, comparacoes);

    char T2[41], P2[] = "aaab";             /* pior caso: 40 letras 'a' */
    memset(T2, 'a', 40);
    T2[40] = '\0';
    int pos2 = casamento(T2, 40, P2, 4);
    printf("pior caso: posicao %d com %ld comparacoes\n", pos2, comparacoes);
    printf("limite m(n-m+1) = %d\n", 4 * (40 - 4 + 1));
    return 0;
}


In [ ]:
# Compila e executa: o pior caso atinge exatamente o limite m(n-m+1)
!gcc -Wall casamento.c -o casamento && ./casamento
!./casamento | grep -q 'texto natural: posicao 7 com 12 comparacoes' \
  && ./casamento | grep -q 'pior caso: posicao -1 com 148 comparacoes' \
  && echo OK || echo 'Verifique: esperava 12 comparacoes no texto natural e 148 no pior caso'


**Observe o contraste:** 12 comparações no texto de 14 letras contra 148 no texto de
40 letras do pior caso. É esse abismo entre caso médio e pior caso que faz o algoritmo
ingênuo sobreviver na prática — e que os algoritmos KMP e Boyer-Moore eliminam ao
**pré-processar o padrão**.

## 2. Mochila 0/1 por busca exaustiva

Cada subconjunto de $\{1,\ldots,n\}$ corresponde a um número de $n$ bits: o bit $i$ ligado
significa "item $i$ está na mochila". Logo, o laço `for (S = 0; S < (1 << n); S++)`
percorre **todo** o espaço de busca, sem esquecer nem repetir nenhum candidato.

Instância dos slides ($W = 10$):

| item | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| peso  | 6  | 5  | 5  | 3  |
| valor | 30 | 21 | 20 | 10 |

In [ ]:
%%writefile mochila.c
#include <stdio.h>

#define N 4
static const int w[N] = {6, 5, 5, 3};
static const int v[N] = {30, 21, 20, 10};
static const int W = 10;

int main(void) {
    int melhor_valor = 0, melhor_S = 0;
    long avaliados = 0;

    for (int S = 0; S < (1 << N); S++) {     /* S e' um subconjunto em binario */
        int peso = 0, valor = 0;
        for (int i = 0; i < N; i++)
            if (S & (1 << i)) {              /* item i pertence a S? */
                peso  += w[i];
                valor += v[i];
            }
        avaliados++;
        if (peso <= W && valor > melhor_valor) {   /* viavel e melhor */
            melhor_valor = valor;
            melhor_S = S;
        }
    }

    printf("subconjuntos avaliados: %ld\n", avaliados);
    printf("melhor valor: %d com itens", melhor_valor);
    for (int i = 0; i < N; i++)
        if (melhor_S & (1 << i)) printf(" %d", i + 1);
    printf("\n");
    return 0;
}


In [ ]:
# Compila e executa: 2^4 = 16 subconjuntos avaliados, otimo = {2,3} com valor 41
!gcc -Wall mochila.c -o mochila && ./mochila
!./mochila | grep -q 'subconjuntos avaliados: 16' \
  && ./mochila | grep -q 'melhor valor: 41 com itens 2 3' \
  && echo OK || echo 'Verifique: esperava 16 subconjuntos e valor otimo 41'


**Experimento rápido:** troque `N` por 25 (com pesos e valores quaisquer) e cronometre.
Depois tente 30, 35... Cada unidade a mais em `N` **dobra** o tempo. É a diferença entre
um algoritmo lento e um algoritmo inviável.

## 3. Caixeiro viajante por busca exaustiva

Fixamos a cidade $0$ como origem e geramos todas as $(n-1)!$ ordens das demais cidades.
A geração usada aqui é recursiva ("escolha quem vem agora entre as ainda não usadas");
métodos sistemáticos de geração — ordem lexicográfica, Johnson-Trotter — são assunto da
próxima semana.

Primeiro validamos com o grafo de 4 cidades dos slides (rota ótima de custo 11);
depois medimos o tempo para $n = 8, \ldots, 12$ (a última linha leva alguns segundos).

In [ ]:
%%writefile tsp.c
#include <stdio.h>
#include <time.h>

#define MAXN 13

static int n;
static int d[MAXN][MAXN];
static int perm[MAXN], usada[MAXN];
static long rotas;
static int melhor;

/* avalia a rota 0 -> perm[1] -> ... -> perm[n-1] -> 0 */
static void avalia(void) {
    int custo = 0, atual = 0;
    for (int k = 1; k < n; k++) {
        custo += d[atual][perm[k]];
        atual = perm[k];
    }
    custo += d[atual][0];
    rotas++;
    if (custo < melhor) melhor = custo;
}

/* completa perm[k..n-1] com as cidades ainda nao usadas */
static void gera(int k) {
    if (k == n) { avalia(); return; }
    for (int c = 1; c < n; c++)
        if (!usada[c]) {
            usada[c] = 1;
            perm[k] = c;
            gera(k + 1);
            usada[c] = 0;                    /* desfaz a escolha */
        }
}

static int resolve(void) {
    rotas = 0;
    melhor = 1 << 30;
    for (int c = 0; c < MAXN; c++) usada[c] = 0;
    gera(1);
    return melhor;
}

int main(void) {
    /* exemplo dos slides: cidades a,b,c,d = 0,1,2,3 */
    int exemplo[4][4] = {{0,2,5,7}, {2,0,8,3}, {5,8,0,1}, {7,3,1,0}};
    n = 4;
    for (int i = 0; i < 4; i++)
        for (int j = 0; j < 4; j++) d[i][j] = exemplo[i][j];
    printf("exemplo: melhor rota custa %d, %ld rotas avaliadas\n", resolve(), rotas);

    /* crescimento fatorial: matriz deterministica, sem numeros aleatorios */
    for (n = 8; n <= 12; n++) {
        for (int i = 0; i < n; i++)
            for (int j = 0; j < n; j++)
                d[i][j] = d[j][i] = (i == j) ? 0 : 1 + ((i * 31 + j * 17 + i * j * 7) % 50);
        clock_t t0 = clock();
        int m = resolve();
        double seg = (double) (clock() - t0) / CLOCKS_PER_SEC;
        printf("n = %2d: %10ld rotas em %7.3f s (melhor = %d)\n", n, rotas, seg, m);
    }
    return 0;
}


In [ ]:
# Compila e executa: observe o tempo sendo multiplicado por (n-1) a cada linha
!gcc -Wall -O2 tsp.c -o tsp && ./tsp
!./tsp | grep -q 'exemplo: melhor rota custa 11, 6 rotas avaliadas' \
  && echo OK || echo 'Verifique: no exemplo dos slides a rota otima custa 11'


**Para responder:** olhando a saída, quanto tempo levaria $n = 15$? E $n = 20$?
(Multiplique o tempo de $n = 12$ por $12 \times 13 \times 14$ e continue.)
Não rode — estime.

## Exercício 1 — Par de pontos mais próximo

Implemente a força bruta $\Theta(n^2)$: examine todos os $\binom{n}{2}$ pares e devolva o
de menor distância. Compare **distâncias ao quadrado** — assim o programa não chama
`sqrt` nenhuma vez e continua devolvendo o mesmo par.

In [ ]:
%%writefile exercicio1.c
#include <stdio.h>

/* Exercicio 1: par de pontos mais proximo por forca bruta.
   Preencha *pi e *pj com os indices do par mais proximo e devolva o
   QUADRADO da distancia entre eles. */

static long par_mais_proximo(int x[], int y[], int n, int *pi, int *pj) {
    /* TODO: percorra todos os pares i < j, calcule dx*dx + dy*dy
             e guarde o menor visto ate agora. */
    *pi = -1; *pj = -1;
    return -1;
}

int main(void) {
    int x[] = { 2, 12, 40,  5, 12,  3};
    int y[] = { 3, 30, 50,  1, 10,  4};
    int n = 6, i, j;
    long d2 = par_mais_proximo(x, y, n, &i, &j);
    printf("par mais proximo: pontos %d e %d (d2 = %ld)\n", i, j, d2);
    return 0;
}


In [ ]:
# Teste automático do Exercício 1
!gcc -Wall exercicio1.c -o exercicio1 && ./exercicio1
!./exercicio1 | grep -q 'par mais proximo: pontos 0 e 5 (d2 = 2)' \
  && echo OK || echo 'Verifique sua implementacao: esperava os pontos 0 e 5 com d2 = 2'


## Exercício 2 — Soma de subconjunto

Dado um vetor de inteiros positivos e um alvo $t$, decidir se **algum** subconjunto soma
exatamente $t$. Use a mesma técnica da mochila: percorra as máscaras de $0$ a $2^n - 1$.

Este é mais um problema NP-completo — a busca exaustiva é a solução exata mais simples
que existe para ele.

In [ ]:
%%writefile exercicio2.c
#include <stdio.h>

/* Exercicio 2: existe subconjunto de A[0..n-1] com soma exatamente igual a alvo?
   Devolva 1 (sim) ou 0 (nao), percorrendo as mascaras de bits. */

static int soma_subconjunto(int A[], int n, int alvo) {
    /* TODO: para cada S de 0 a (1<<n)-1, some os elementos com bit ligado
             e compare com alvo. */
    return 0;
}

int main(void) {
    int A[] = {3, 34, 4, 12, 5, 2};
    int n = 6;
    int alvos[] = {9, 30, 26, 1};
    for (int k = 0; k < 4; k++)
        printf("soma %d: %s\n", alvos[k],
               soma_subconjunto(A, n, alvos[k]) ? "sim" : "nao");
    return 0;
}


In [ ]:
# Teste automático do Exercício 2
!gcc -Wall exercicio2.c -o exercicio2 && ./exercicio2
!./exercicio2 | grep -q 'soma 9: sim' && ./exercicio2 | grep -q 'soma 30: nao' \
  && ./exercicio2 | grep -q 'soma 26: sim' && ./exercicio2 | grep -q 'soma 1: nao' \
  && echo OK || echo 'Verifique sua implementacao: esperava sim, nao, sim, nao'


### Para pensar (responda em uma célula de texto)

1. No casamento ingênuo, por que o pior caso precisa de um padrão que quase casa em
   toda posição? Construa um par $(T, P)$ com $n = 20$ e $m = 5$ que atinja o limite.
2. A mochila exaustiva soma cada subconjunto do zero, gastando $\Theta(n)$ por candidato.
   Como reaproveitar a soma do subconjunto anterior mudaria a análise? Qual seria a
   ordem de crescimento total?
3. O problema da atribuição tem $n!$ candidatos, mas é resolvido em $O(n^3)$ pelo método
   húngaro. Isso contradiz o argumento de que "há $n!$ possibilidades, logo é inviável"?
   O que exatamente esse argumento prova — e o que ele **não** prova?

## Desafio Final — A força bruta como oráculo

O algoritmo guloso para a mochila 0/1 ordena os itens pela razão $v_i / w_i$ e pega cada
item que ainda couber. Ele é rápido, intuitivo... e **errado** (não devolve sempre o ótimo).

Sua tarefa: provar isso experimentalmente, sem procurar contraexemplo no papel.

1. Implemente `guloso_por_razao` e `exaustivo` (reaproveite a máscara de bits da Seção 2);
2. Gere instâncias pseudoaleatórias pequenas ($n \le 12$, pesos e valores até 20,
   $W$ até a metade do peso total);
3. Rode as duas e pare no primeiro caso em que os valores diferem;
4. Imprima a instância e os dois valores — esse é o seu contraexemplo.

Depois responda: o guloso pode devolver um valor **maior** que o exaustivo? Por quê?

In [ ]:
%%writefile desafio.c
#include <stdio.h>
#include <stdlib.h>

#define MAXN 12

/* Desafio: usar a mochila exaustiva como oraculo do algoritmo guloso. */

static int exaustivo(int w[], int v[], int n, int W) {
    /* TODO: percorra as 2^n mascaras e devolva o melhor valor viavel. */
    return 0;
}

static int guloso_por_razao(int w[], int v[], int n, int W) {
    /* TODO: ordene os indices por v[i]/w[i] decrescente (compare
             v[i]*w[j] > v[j]*w[i] para evitar ponto flutuante) e pegue
             cada item que ainda couber. Devolva o valor total obtido. */
    return 0;
}

int main(void) {
    srand(20260827);                       /* semente fixa: experimento reproduzivel */
    for (long teste = 1; teste <= 100000; teste++) {
        int n = 4 + rand() % (MAXN - 3);
        int w[MAXN], v[MAXN], soma = 0;
        for (int i = 0; i < n; i++) {
            w[i] = 1 + rand() % 20;
            v[i] = 1 + rand() % 20;
            soma += w[i];
        }
        int W = 1 + rand() % (soma / 2 + 1);

        int otimo = exaustivo(w, v, n, W);
        int aprox = guloso_por_razao(w, v, n, W);

        if (otimo != aprox) {
            printf("contraexemplo no teste %ld: n = %d, W = %d\n", teste, n, W);
            for (int i = 0; i < n; i++)
                printf("  item %d: peso %2d valor %2d\n", i + 1, w[i], v[i]);
            printf("exaustivo = %d, guloso = %d\n", otimo, aprox);
            return 0;
        }
    }
    printf("nenhuma divergencia encontrada (suspeite do seu exaustivo!)\n");
    return 0;
}


In [ ]:
# Compile e rode o seu desafio
!gcc -Wall desafio.c -o desafio && ./desafio


## Referências

Veja o arquivo `../referencias.bib` para a lista completa. O capítulo 3 de Levitin é
dedicado inteiramente a força bruta e busca exaustiva.
